In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from data_prep import ColumnSelector
from unlabeled_lr import UnlabeledLogReg
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


In [84]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
polish_companies_bankruptcy = fetch_ucirepo(id=365) 
  
# data (as pandas dataframes) 
X = polish_companies_bankruptcy.data.features 
y = polish_companies_bankruptcy.data.targets 
  


In [85]:
breast_cancer_coimbra = fetch_ucirepo(id=451)

X = breast_cancer_coimbra.data.features
y = breast_cancer_coimbra.data.targets

In [86]:
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.3)

In [87]:
X_train

,Age,BMI,Glucose,Insulin,HOMA,Leptin,Adiponectin,Resistin,MCP.1
107,46,33.180000,92,5.750,1.304867,18.6900,9.160000,8.89000,209.190
35,67,29.606767,79,5.819,1.133929,21.9033,2.194280,4.20750,585.307
86,48,28.125000,90,2.540,0.563880,15.5325,10.222310,16.11032,1698.440
54,49,20.956608,94,12.305,2.853119,11.2406,8.412175,23.11770,573.630
49,85,26.600000,96,4.462,1.056602,7.8500,7.931700,9.61350,232.006
...,...,...,...,...,...,...,...,...,...
111,45,26.850000,92,3.330,0.755688,54.6800,12.100000,10.96000,268.230
95,49,29.777778,70,8.396,1.449709,51.3387,10.731740,20.76801,602.486
52,45,21.303949,102,13.852,3.485163,7.6476,21.056625,23.03408,552.444
57,68,21.082813,102,6.200,1.559920,9.6994,8.574655,13.74244,448.799


In [88]:

pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("selector", ColumnSelector(threshold=0.7)),
    ]
)


In [89]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()), ('selector', ColumnSelector())])

In [90]:
X_train = pipeline.transform(X_train)

In [91]:
y_train

,Classification
107,2
35,1
86,2
54,2
49,1
...,...
111,2
95,2
52,2
57,2


In [92]:
from missing import *

In [93]:
y_train["Y_observed"] = y_train["Classification"].copy().astype(int)
y_train["S"] = 0
y_train["Missing_Y"] = "no"

In [94]:
y_train_missing = MCAR(y_train)

In [95]:
y_train_missing

,Classification,Y_observed,S,Missing_Y
107,2,-1,1,yes
35,1,1,0,no
86,2,2,0,no
54,2,2,0,no
49,1,1,0,no
...,...,...,...,...
111,2,2,0,no
95,2,2,0,no
52,2,2,0,no
57,2,-1,1,yes


In [96]:
y_train_missing_one_col = y_train_missing["Y_observed"]

In [97]:
ulr = UnlabeledLogReg(y_imputation_method="random")
ulr.fit(X_train, y_train_missing_one_col)

/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: overflow encountered in exp
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: invalid value encountered in divide
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


In [98]:
X_val = pipeline.transform(X_val)

In [104]:
ulr.validate(X_val, y_val, measure="f1")

/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: overflow encountered in exp
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: invalid value encountered in divide
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: overflow encountered in exp
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWarning: invalid value encountered in divide
  return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
/Users/julia/Desktop/semestr-8/aml/projects/semi-supervised-FISTA-logistic-lasso/src/utils.py:50: RuntimeWar

(0.5955882352941176, 0.5955882352941176)

In [105]:
y_train_missing_one_col.value_counts()

Y_observed
 2    35
 1    30
-1    16
Name: count, dtype: int64

In [106]:
ulr.model.all_validation_scores_[0][:5]

[0.6274509803921569,
 0.6274509803921569,
 0.6274509803921569,
 0.6274509803921569,
 0.6274509803921569]